# 🦠 Viral Mutation & Immune Escape Prediction
### Multimodal Deep Learning (ESM-2 + GVP-GNN + Cross-Attention)

**Hardware**: Google Colab Free T4 GPU  
**Goal**: Train joint model on Spike protein sequences & 3D contact graphs, then evaluate prospective risk prediction on held-out variants (`KP.3`, `XEC`).

## 1. Setup Environment & GPU Check

In [ ]:
!nvidia-smi
!pip install -q torch-geometric torch-scatter torch-sparse fair-esm biopython zstandard freesasa xgboost scikit-learn pandas scipy pyyaml

## 2. Upload Data or Zip Project
Upload `viral-escape-predictor.zip` from your local machine to Colab and unzip it.

In [ ]:
!unzip -q viral-escape-predictor.zip
%cd viral-escape-predictor
print("Project directory ready!")

## 3. Extract ESM-2 650M Embeddings (Free T4 GPU)
Runs inference pass over 1,326 Spike sequences with `esm2_t33_650M_UR50D` in `fp16` and saves `.pt` embedding tensors to disk.

In [ ]:
!python src/features/esm2_embed.py

## 4. Train Joint Multimodal Model (GVP-GNN + Cross-Attention)
Trains dual prediction heads (Hotspot BCE + Escape MSE) using joint loss on free T4 GPU.

In [ ]:
!python src/train.py --config_data configs/data.yaml --config_model configs/model.yaml --config_train configs/train.yaml

## 5. Prospective Evaluation & Backtesting
Evaluates trained checkpoint on held-out variants (`KP.3`, `XEC`) and outputs Spearman $\rho$, MSE, AUROC, AUPRC, and Precision@5/10/20.

In [ ]:
!python src/eval.py --checkpoint checkpoints/best_model.pt

## 6. Run 4-Variant Ablation Study
Compares: (1) Seq-only, (2) Struct-only, (3) Concat Baseline, and (4) Full Cross-Attention Model.

In [ ]:
!python src/ablations.py